In [ ]:
import sys
from pathlib import Path
import pandas as pd
import seaborn as sns

sys.path.insert(0, '/home/rzhu/Desktop/projects/kinase_analysis/src/')
from funcs_featurise import *
from funcs_db_assign import *
from funcs_indices import *
from funcs_plotting import *
from TrajData import TrajData
from MSMStudy import MSMStudy

In [ ]:
protein = 'abl'
key = 'abl-pdb-50ps' # Dataset key 
study = 'scan_lags' # where the results are saved
data_path = Path(f'/home/rzhu/Desktop/projects/kinase_analysis/data') # base dir

hps_df = pd.read_csv(data_path/f'{protein}'/'msm'/f'{study}'/'hps.csv')
TD = TrajData(protein = protein)
TD.add_dataset(rtraj_dir = Path(f'/arc/abl_processed/'), 
               ftraj_dir= data_path / f'{protein}'/ f'{key}' / 'ftrajs',
               dt=0.05,
               key=key)
study = MSMStudy(hps_table = hps_df,
                 traj_data = TD,
                 wk_dir = data_path/f'{protein}'/'msm'/f'{study}')
study.observation

In [ ]:
study.set_hp_id(11)

In [ ]:
study.run_pcca(6)

In [ ]:
inactive_ms_id = 5
active_ms_id = 4

start_state_ids = np.where(study.pcca_mod.memberships[:,inactive_ms_id] > 0.95)[0]
end_state_ids = np.where(study.pcca_mod.memberships[:,active_ms_id] > 0.95)[0]
len(start_state_ids), len(end_state_ids)

In [ ]:
tpt_activation = study.msm_mod.reactive_flux(target_states=end_state_ids, source_states=start_state_ids)
paths, pathfluxes = tpt_activation.pathways(fraction=1.0, maxiter=500)

In [ ]:
len(paths), len(pathfluxes)

* Plot top-100 paths by path capacity (path flux)
* Plot only microstates involved in the paths colored by forward committor

In [ ]:
import matplotlib.patheffects as pe
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

contour_cmap = sns.cubehelix_palette(
    n_colors=32,
    start=2,
    rot=0.25,
    gamma=1.4,
    hue=1.1,
    light=1,
    dark=0.1,
    as_cmap=True
)

traj_all = study.ttraj_cat
traj_weights = np.concatenate(study.traj_weights, axis=0)
c_centers = study.kmeans_centers[study.connected_states, :]

# Log transform flux capacities to highlight dominant paths
n_paths = 100
lw_min, lw_max = 0.1, 2
microstates_in_top_n = np.unique(np.concatenate(paths[:n_paths])).astype(int)
flux = pathfluxes[:n_paths]
fmax, fmin = np.max(flux), np.min(flux)
flux_log = (np.log(flux) - np.log(fmin)) / (np.log(fmax) - np.log(fmin))
fws = lw_min + (lw_max - lw_min) * flux_log

# Landscape and microstates
fig, ax = plt.subplots(figsize=(7, 6))
ax, contour, cbar = plot_energy2d(energy2d(traj_all[:, 0], traj_all[:, 1], weights=traj_weights), ax=ax, contourf_kws=dict(cmap=contour_cmap))
sc = ax.scatter(
    c_centers[microstates_in_top_n,0],
    c_centers[microstates_in_top_n,1],
    c=tpt_activation.forward_committor[microstates_in_top_n],
    vmin=0, vmax=1,
    cmap="YlOrRd",
    s=30,
    edgecolors="k",      # black edge
    linewidths=0.5       # thin outline so points stay visible
)

# Add an inset colorbar
cax_sc = inset_axes(ax, width="30%", height="3%", loc="upper left", borderpad=2)
cbar_sc = fig.colorbar(sc, cax=cax_sc, orientation="horizontal")
cbar_sc.set_label("Forward committor", fontsize=12)

# Plot pathways
for path, fw in zip(paths[:n_paths], fws):
    path = np.asarray(path, dtype=int)
    ax.plot(c_centers[path,0], c_centers[path,1], alpha=0.2, linewidth=fw, c='k')

# index of thickest (largest linewidth)
imax = int(np.argmax(fws))                     
p_thick = np.asarray(paths[imax], dtype=int)

# draw the thickest path a bit darker on top (optional but helps visibility)
ax.plot(c_centers[p_thick,0], c_centers[p_thick,1],
        alpha=0.9, linewidth=float(fws[imax]), c='k', zorder=5)

lw = float(fws[imax])
ms = 12
edge_color = "w"
main_color = "k"

for a, b in zip(p_thick[:-1], p_thick[1:]):
    x0, y0 = c_centers[a, 0], c_centers[a, 1]
    x1, y1 = c_centers[b, 0], c_centers[b, 1]

    ann = ax.annotate(
        "",
        xy=(x1, y1), xytext=(x0, y0),
        arrowprops=dict(
            arrowstyle="-|>",
            color=main_color,
            lw=lw,
            mutation_scale=ms,
            shrinkA=3, shrinkB=2,
            capstyle="round",
            joinstyle="round",
        ),
        zorder=8
    )

    ann.arrow_patch.set_path_effects([
        pe.Stroke(linewidth=lw + 2.0, foreground=edge_color),
        pe.Normal(),
    ])

ax.set_xlabel('tIC1', fontsize=14)
ax.set_ylabel('tIC2', fontsize=14)
cbar.ax.set_ylabel('Free energy (kT)', fontsize=14)

# plt.savefig('abl_flux_capacities.pdf')
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

contour_cmap = sns.cubehelix_palette(
    n_colors=32,
    start=2,
    rot=0.25,
    gamma=1.4,
    hue=1.1,
    light=1,
    dark=0.1,
    as_cmap=True
)

traj_all = study.ttraj_cat
traj_weights = np.concatenate(study.traj_weights, axis=0)
c_centers = study.kmeans_centers[study.connected_states, :]

# Log transform flux capacities to highlight dominant paths
n_paths = 100
lw_min, lw_max = 0.1, 2
microstates_in_top_n = np.unique(np.concatenate(paths[:n_paths])).astype(int)
flux = pathfluxes[:n_paths]
fmax, fmin = np.max(flux), np.min(flux)
flux_log = (np.log(flux) - np.log(fmin)) / (np.log(fmax) - np.log(fmin))
fws = lw_min + (lw_max - lw_min) * flux_log

# Landscape and microstates
fig, ax = plt.subplots(figsize=(7, 6))
ax, contour, cbar = plot_energy2d(
    energy2d(traj_all[:, 0], traj_all[:, 1], weights=traj_weights),
    ax=ax,
    contourf_kws=dict(cmap=contour_cmap)
)

sc = ax.scatter(
    c_centers[microstates_in_top_n, 0],
    c_centers[microstates_in_top_n, 1],
    c=tpt_activation.forward_committor[microstates_in_top_n],
    vmin=0, vmax=1,
    cmap="YlOrRd",
    s=30,
    edgecolors="k",
    linewidths=0.5
)

# Add an inset colorbar
cax_sc = inset_axes(ax, width="30%", height="3%", loc="upper left", borderpad=2)
cbar_sc = fig.colorbar(sc, cax=cax_sc, orientation="horizontal")
cbar_sc.set_label("Forward committor", fontsize=12)

# Thickest path index
imax = int(np.argmax(fws))
p_thick = np.asarray(paths[imax], dtype=int)

# Curvature pattern
rads = [0.18, -0.18, 0.18, -0.18]

# Draw all non-thickest pathways as curved lines (no arrowheads)
for ip, (path, fw) in enumerate(zip(paths[:n_paths], fws)):
    if ip == imax:
        continue

    path = np.asarray(path, dtype=int)

    for i, (a, b) in enumerate(zip(path[:-1], path[1:])):
        x0, y0 = c_centers[a, 0], c_centers[a, 1]
        x1, y1 = c_centers[b, 0], c_centers[b, 1]
        rad = rads[i % len(rads)]

        curved_line = FancyArrowPatch(
            (x0, y0), (x1, y1),
            arrowstyle="-",                      # line only, no arrowhead
            connectionstyle=f"arc3,rad={rad}",
            linewidth=float(fw),
            color="k",
            alpha=0.2,
            capstyle="round",
            joinstyle="round",
            zorder=3
        )
        ax.add_patch(curved_line)

# Draw thickest path as curved arrows
lw = float(fws[imax])
ms = 12
edge_color = "w"
main_color = "k"

for i, (a, b) in enumerate(zip(p_thick[:-1], p_thick[1:])):
    x0, y0 = c_centers[a, 0], c_centers[a, 1]
    x1, y1 = c_centers[b, 0], c_centers[b, 1]
    rad = rads[i % len(rads)]

    arrow = FancyArrowPatch(
        (x0, y0), (x1, y1),
        arrowstyle="-|>",
        connectionstyle=f"arc3,rad={rad}",
        mutation_scale=ms,
        linewidth=lw,
        color=main_color,
        shrinkA=3,
        shrinkB=2,
        capstyle="round",
        joinstyle="round",
        zorder=8
    )
    arrow.set_path_effects([
        pe.Stroke(linewidth=lw + 2.0, foreground=edge_color),
        pe.Normal(),
    ])
    ax.add_patch(arrow)

ax.set_xlabel("tIC1", fontsize=14)
ax.set_ylabel("tIC2", fontsize=14)
cbar.ax.set_ylabel("Free energy (kT)", fontsize=14)

plt.savefig("abl_flux_capacities.pdf")
plt.show()